# Setup
This Nodebook should help generate the input files used for the mammalian cell culture simulation. 

In [1]:
from dbiolab.mammalian_cellculture_simulation import (
    CellType,
    FeedConcentrations,
    FeedingEvent,
    MammalianCellcultureReactor,
    MammalianRecipe,
    NoiseParameters,
    ReactorType,
    plot_reaction_evolution_timeseries,
    plot_setpoints,
)

## 1. Define Reactor Type

In [ ]:
reactor = ReactorType(
    name="MTP_1",
    volume=2.0,
    scale_factor=1.0,
    growth_inhibition=1.0,
    # 2 L stirred tank, pitched-blade impeller, ~0.05 vvm air (van't Riet kLa)
    tank_diameter=0.13,
    impeller_diameter=0.05,
    power_number=1.5,
    superficial_gas_velocity=1.25e-4,
    kla_constant=0.002,
    kla_power_exponent=0.7,
    kla_gas_exponent=0.2,
    max_to_mean_dissipation=30.0,
    o2_saturation=0.2,
    gas_co2_fraction=0.05,  # CO2 in the sparged gas, for pH control
)

## 2. Define Cell Type

In [ ]:
celltype = CellType(
    name="celltype_1_alpha",
    mu_max=0.0429057935,
    k_lysis=0.01,
    k_deg_gln=0.0013441722,
    k_death_max=0.0035,
    k_death_mu=0.0145125692,
    k_sat_lac=25.0,
    k_sat_ammonia=10.0,
    k_sat_glc=1.1634941712,
    k_sat_gln=0.2123961928,
    y_ammonia=0.8,
    y_lactate=0.8061528449,
    y_cells_glc=3000000.0,
    y_cells_gln=15000000.0,
    m_glc=2e-09,
    m_gln=0.0,
    ph_opt=7.1,
    ph_robustness_factor=1.0,
    steepness_factor_ph_l=0.8,
    steepness_factor_ph_r=0.8,
    smoothness_factor_ph_l=0.6,
    smoothness_factor_ph_r=0.6,
    e_activation=36.5,
    temperature_robustness_factor=1.0,
    steepness_factor_t_l=0.2,
    steepness_factor_t_r=0.7,
    smoothness_factor_t_l=3.0,
    smoothness_factor_t_r=8.0,
    y_product_non_growth=6e-11,
    y_product_from_viable=3e-09,
    hcp_per_cell=1e-09,
    k_agg_product=0.03,
    y_glc_product=200.0,
    ph_opt_product=6.9,
    ph_robustness_factor_product=1.0,
    steepness_factor_ph_l_product=0.8,
    steepness_factor_ph_r_product=0.8,
    smoothness_factor_ph_l_product=0.3,
    smoothness_factor_ph_r_product=0.3,
    y_lactate_max=1.8,
    k_overflow_glc=10.0,
    k_inh_lac_product=30.0,
    k_death_lac=10.0,
    y_gln_product=50.0,
    k_inh_ammonia_product=15.0,
    k_death_ammonia=10.0,
    ph_min_factor_product=0.05,
    k_death_stress=4.0,
    t_opt_product=32.0,
    temperature_robustness_factor_product=1.0,
    steepness_factor_t_l_product=0.5,
    steepness_factor_t_r_product=0.5,
    smoothness_factor_t_l_product=3.0,
    smoothness_factor_t_r_product=3.0,
    q_lac_uptake=1e-08,
    k_glc_lac_switch=1.0,
    k_sat_lac_uptake=2.0,
    k_glycation=0.0001,
    k_lactate_ph=1.0,
    ph_ref_lactate=7.0,
    q10_metabolism=2.0,
    t_ref_metabolism=37.0,
    q_o2=1e-07,
    k_sat_o2=0.01,
    k_lactate_hypoxia=1.0,
    eps_crit=5.0,
    t_mix_crit=90.0,
    respiratory_quotient=0.95,
    pco2_crit=150.0,
)

## 3. Define Recipe

### Initial Conditions
The best recipe found for the ddo-bench `mammalian_fed_batch` problem (by differential evolution): little glucose and glutamine at the start, to keep lactate and ammonia low.

In [ ]:
initial_conditions = FeedConcentrations(
    c_product=0.0,
    x_total=800000.0,
    x_viable=800000.0,
    x_dead=0.0,
    c_glc=5.02,
    c_gln=1.27,
    c_lac=0.0,
    c_ammonia=0.0,
    c_aggregates=0.0,
    c_glycated=0.0,
    c_hcp=0.0,
    c_o2=0.2,  # saturated with air
    c_co2=1.228,  # 40 mmHg, medium equilibrated with 5 % CO2
)

### Feed Events
Three bolus feeds of glucose and glutamine. Each feed also sets temperature and pH: grow near the growth optimum (about 36 C), then step the temperature down to mild hypothermia (33 C) for production, which also slows metabolism and lactate formation. pH is kept near 6.9: production prefers a slightly low pH, and a higher pH makes more lactate. The feed volume follows from the stock solutions (1000 mM glucose, 200 mM glutamine) and dilutes the culture. Stirring trades off mixing against shear: too slow and cells pass through zones of concentrated feed and low oxygen, too fast and the impeller damages them. Dissolved oxygen is held at 40 % air saturation by enriching the sparged gas with pure oxygen.

In [ ]:
GLUCOSE_STOCK = 1000.0  # mM
GLUTAMINE_STOCK = 200.0  # mM


def make_feed(time_point, c_glc, c_gln, temperature, ph_value, stirring_rate):
    """A glucose/glutamine bolus that also sets temperature, pH and stirring."""
    return FeedingEvent(
        time_point=time_point,
        feed_concentrations=FeedConcentrations(
            c_product=0.0,
            x_total=0.0,
            x_viable=0.0,
            x_dead=0.0,
            c_glc=c_glc,
            c_gln=c_gln,
            c_lac=0.0,
            c_ammonia=0.0,
            c_aggregates=0.0,
            c_glycated=0.0,
            c_hcp=0.0,
            c_o2=0.0,
            c_co2=0.0,
        ),
        volume_fraction=c_glc / GLUCOSE_STOCK + c_gln / GLUTAMINE_STOCK,
        temperature=temperature,
        ph_value=ph_value,
        stirring_rate=stirring_rate,
    )


feeding_strategy = [
    make_feed(72.0, 1.89, 1.1, temperature=35.68, ph_value=6.86, stirring_rate=314.64),
    make_feed(144.0, 12.39, 4.71, temperature=34.4, ph_value=6.85, stirring_rate=349.25),
    make_feed(216.0, 20.0, 4.97, temperature=33.11, ph_value=6.92, stirring_rate=385.89),
]

### Noise in Simulations

In [6]:
noise_parameters = NoiseParameters(
    measurement_noise_c_product=0.05,
    measurement_noise_x_total=0.05,
    measurement_noise_x_viable=0.05,
    measurement_noise_x_dead=0.05,
    measurement_noise_c_glc=0.05,
    measurement_noise_c_gln=0.05,
    measurement_noise_c_lac=0.05,
    measurement_noise_c_ammonia=0.05,
    measurement_noise_c_aggregates=0.05,
)
noise_parameters = NoiseParameters()

## 4. Run Simulation

In [ ]:
# You can load the saved objects from JSON files

# celltype = CellType.from_json(filepaths["celltype"])
# initial_conditions = FeedConcentrations.from_json(filepaths["initial_conditions"])
# noise_parameters = NoiseParameters.from_json(filepaths["noise"])

simulator = MammalianCellcultureReactor(reactor_type=reactor)
recipe = MammalianRecipe(
    name="Best recipe",
    celltype=celltype,
    temperature=36.13,  # range 30-40, until the first feed
    ph_value=6.89,  # range 6.5-8, until the first feed
    stirring_rate=279.78,  # rpm, range 50-600, until the first feed
    initial_conditions=initial_conditions,
    feeding_strategy=feeding_strategy,
    runtime=336.0,
    measurement_intervals=12.0,
    noise_parameters=noise_parameters,
    do_setpoint=0.4,  # 40 % air saturation, held by oxygen enrichment
)
solution = simulator.run_experiment(recipe=recipe)

## 5. Visualize Results

In [8]:
solution_plots = plot_reaction_evolution_timeseries(solution)
for fig in solution_plots:
    fig.show()

### Setpoint tracking
The temperature and pH setpoints the culture runs at, switched at the feeds (dotted lines).

In [9]:
plot_setpoints(recipe).show()

## 6. A Bad Recipe for Comparison
A recipe that looks generous but breaks every rule above: constant 37 C and pH 7.4, gentle stirring (100 rpm) with air only (no oxygen enrichment), so a dense culture runs out of oxygen and mixing is poor (far from the production pH, and a high pH makes more lactate), and as much glucose and glutamine as possible at the start and at every feed. Dissolved oxygen collapses, the excess glucose turns into lactate and glycates the product, the excess glutamine into ammonia.

In [ ]:
bad_initial_conditions = FeedConcentrations(
    c_product=0.0,
    x_total=800000.0,
    x_viable=800000.0,
    x_dead=0.0,
    c_glc=30.0,
    c_gln=8.0,
    c_lac=0.0,
    c_ammonia=0.0,
    c_aggregates=0.0,
    c_glycated=0.0,
    c_hcp=0.0,
    c_o2=0.2,  # saturated with air
    c_co2=1.228,  # 40 mmHg, medium equilibrated with 5 % CO2
)
bad_recipe = MammalianRecipe(
    name="Bad recipe",
    celltype=celltype,
    temperature=37.0,
    ph_value=7.4,
    stirring_rate=100.0,
    initial_conditions=bad_initial_conditions,
    feeding_strategy=[
        make_feed(t, 20.0, 5.0, temperature=37.0, ph_value=7.4, stirring_rate=100.0)
        for t in (72.0, 144.0, 216.0)
    ],
    runtime=336.0,
    measurement_intervals=12.0,
    noise_parameters=noise_parameters,
    do_setpoint=None,  # air only, no oxygen enrichment
)
bad_solution = simulator.run_experiment(recipe=bad_recipe)

In [11]:
for fig in plot_reaction_evolution_timeseries(bad_solution):
    fig.show()
plot_setpoints(bad_recipe).show()

### Comparison at Harvest

In [ ]:
for name, result in [("Best", solution), ("Bad", bad_solution)]:
    print(
        f"{name:>4}: product {result.c_product[-1]:.4f} mM, "
        f"glycated {result.c_glycated[-1]:.4f} mM, "
        f"lactate {result.c_lac[-1]:.1f} mM, "
        f"ammonia {result.c_ammonia[-1]:.2f} mM, "
        f"min. DO {result.c_o2.min() / reactor.o2_saturation:.0%}, "
        f"purity {result.purity[-1]:.0%}"
    )